In [ ]:
from google.colab import drive
import os
import json
import glob

In [ ]:
# 1. Mount Google Drive and set MegaLoc Log Directory
drive.mount('/content/drive')
drive_log_dir = "/content/drive/MyDrive/VPR_megaloc_logs"
os.makedirs(drive_log_dir, exist_ok=True)

# 2. Clone repo
if not os.path.exists("Visual-Place-Recognition-Project"):
    !git clone --recursive https://github.com/AxelBadouel/Visual-Place-Recognition-Project.git
%cd Visual-Place-Recognition-Project

# 3. Install dependencies
!pip install -q -r requirements.txt
!pip install -q -r image-matching-models/requirements.txt
!pip install -q faiss-cpu

# 4. Download datasets
!python download_datasets.py

Mounted at /content/drive
Cloning into 'Visual-Place-Recognition-Project'...
remote: Enumerating objects: 159, done.
remote: Counting objects: 100% (47/47), done.
remote: Compressing objects: 100% (36/36), done.
remote: Total 159 (delta 23), reused 11 (delta 11), pack-reused 112 (from 4)
Receiving objects: 100% (159/159), 1.38 MiB | 3.79 MiB/s, done.
Resolving deltas: 100% (33/33), done.
Submodule 'image-matching-models' (https://github.com/alexstoken/image-matching-models.git) registered for path 'image-matching-models'
Cloning into '/content/Visual-Place-Recognition-Project/image-matching-models'...
remote: Enumerating objects: 2853, done.        
remote: Counting objects: 100% (1247/1247), done.        
remote: Compressing objects: 100% (377/377), done.        
remote: Total 2853 (delta 987), reused 886 (delta 868), pack-reused 1606 (from 1)        
Receiving objects: 100% (2853/2853), 84.38 MiB | 22.17 MiB/s, done.
Resolving deltas: 100% (2024/2024), done.
Submodule path 'image-mat

# 1. VPR Evaluation

In [ ]:
# 5. Dataset splits definition
test_set_db_queries = [
    ("sf_xs",       "/content/Visual-Place-Recognition-Project/data/sf_xs/test/database",       "/content/Visual-Place-Recognition-Project/data/sf_xs/test/queries"),
    ("tokyo_xs",    "/content/Visual-Place-Recognition-Project/data/tokyo_xs/test/database",    "/content/Visual-Place-Recognition-Project/data/tokyo_xs/test/queries"),
    ("svox_night",  "/content/Visual-Place-Recognition-Project/data/svox/images/test/gallery",  "/content/Visual-Place-Recognition-Project/data/svox/images/test/queries_night"),
    ("svox_sun",    "/content/Visual-Place-Recognition-Project/data/svox/images/test/gallery",  "/content/Visual-Place-Recognition-Project/data/svox/images/test/queries_sun"),
]

# Track run directories
mapping_file = f"{drive_log_dir}/dataset_run_mapping.json"
mapping = json.load(open(mapping_file)) if os.path.exists(mapping_file) else {}

for name, database, queries in test_set_db_queries:
    if name in mapping and os.path.isdir(f"{mapping[name]}/preds") and len(os.listdir(f"{mapping[name]}/preds")) > 0:
        print(f"[STATUS] Skipping Stage 1 for {name}: predictions already cached at {mapping[name]}/preds")
        continue

    print(f"\n==================================================================")
    print(f"Executing MegaLoc Retrieval on Benchmark Dataset: {name}")
    print(f"==================================================================")

    before_runs = set(glob.glob(f"{drive_log_dir}/*"))

    !python VPR-methods-evaluation/main.py \
        --num_workers 4 \
        --batch_size 16 \
        --log_dir {drive_log_dir} \
        --method="megaloc" \
        --backbone="Dinov2" \
        --descriptors_dimension=8448 \
        --image_size 518 518 \
        --database_folder {database} \
        --queries_folder {queries} \
        --num_preds_to_save 20 \
        --recall_values 1 5 10 20 \
        --distance_metric "L2"

    after_runs = set(glob.glob(f"{drive_log_dir}/*"))
    new_runs = after_runs - before_runs
    if new_runs:
        mapping[name] = sorted(new_runs)[-1]
        with open(mapping_file, "w") as f:
            json.dump(mapping, f, indent=2)
        print(f"[METADATA] Successfully mapped dataset '{name}' -> {mapping[name]}")

print("\n[SUMMARY] Active Dataset-to-Directory Mapping for MegaLoc:")
print(json.dumps(mapping, indent=2))

[STATUS] Skipping Stage 1 for sf_xs: predictions already cached at /content/drive/MyDrive/VPR_megaloc_logs/2026-08-22_15-54-43/preds
[STATUS] Skipping Stage 1 for tokyo_xs: predictions already cached at /content/drive/MyDrive/VPR_megaloc_logs/2026-08-22_16-47-53/preds
[STATUS] Skipping Stage 1 for svox_night: predictions already cached at /content/drive/MyDrive/VPR_megaloc_logs/2026-08-22_17-11-49/preds
[STATUS] Skipping Stage 1 for svox_sun: predictions already cached at /content/drive/MyDrive/VPR_megaloc_logs/2026-08-22_17-45-58/preds

[SUMMARY] Active Dataset-to-Directory Mapping for MegaLoc:
{
  "sf_xs": "/content/drive/MyDrive/VPR_megaloc_logs/2026-08-22_15-54-43",
  "tokyo_xs": "/content/drive/MyDrive/VPR_megaloc_logs/2026-08-22_16-47-53",
  "svox_night": "/content/drive/MyDrive/VPR_megaloc_logs/2026-08-22_17-11-49",
  "svox_sun": "/content/drive/MyDrive/VPR_megaloc_logs/2026-08-22_17-45-58"
}


# 2. Image Matching & Re-Ranking on Retrieval Results

In [ ]:
matchers = ['superpoint-lg', 'superglue', 'loftr']

for name, log_dir in mapping.items():
    preds_dir = f"{log_dir}/preds"
    if not os.path.isdir(preds_dir):
        print(f"[WARNING] Missing Stage 1 candidates for {name} at {preds_dir}. Skipping.")
        continue

    for matcher in matchers:
        target_inliers_folder = f"{preds_dir}_{matcher}"

        if os.path.exists(target_inliers_folder) and len(os.listdir(target_inliers_folder)) > 0:
            print(f"[STATUS] Skipping {matcher.upper()} on {name}: inliers already computed.")
            continue

        print(f"\n==================================================================")
        print(f"STAGE 2 MATCHING: Running {matcher.upper()} on Dataset: {name}")
        print(f"Target Predictions Directory: {preds_dir}")
        print(f"==================================================================")

        !python match_queries_preds.py \
            --preds-dir {preds_dir} \
            --matcher {matcher} \
            --device 'cuda' \
            --num-preds 20

[STATUS] Skipping SUPERPOINT-LG on sf_xs: inliers already computed.
[STATUS] Skipping SUPERGLUE on sf_xs: inliers already computed.
[STATUS] Skipping LOFTR on sf_xs: inliers already computed.
[STATUS] Skipping SUPERPOINT-LG on tokyo_xs: inliers already computed.
[STATUS] Skipping SUPERGLUE on tokyo_xs: inliers already computed.

STAGE 2 MATCHING: Running LOFTR on Dataset: tokyo_xs
Target Predictions Directory: /content/drive/MyDrive/VPR_megaloc_logs/2026-08-22_16-47-53/preds
/content/Visual-Place-Recognition-Project/image-matching-models/matching/third_party/LightGlue/lightglue/lightglue.py:24: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_fwd(cast_inputs=torch.float32)
100% 44.2M/44.2M [00:02<00:00, 15.8MB/s]
100% 315/315 [22:14<00:00,  4.24s/it]

STAGE 2 MATCHING: Running SUPERPOINT-LG on Dataset: svox_night
Target Predictions Directory: /content/drive/MyDrive/VPR_meg

In [ ]:
for name, log_dir in mapping.items():
    preds_dir = f"{log_dir}/preds"
    inliers_dirs = f"{preds_dir}_loftr {preds_dir}_superglue {preds_dir}_superpoint-lg"

    print(f"\n==================================================================")
    print(f"FINAL RE-RANKING SCORES (MegaLoc): {name.upper()}")
    print(f"Base Directory: {log_dir}")
    print(f"==================================================================")

    !python reranking.py \
        --preds_dir {preds_dir} \
        --inliers_dir {inliers_dirs} \
        --num-preds 20 \
        --recall-values 1 5 10 20


FINAL RE-RANKING SCORES (MegaLoc): SF_XS
Base Directory: /content/drive/MyDrive/VPR_megaloc_logs/2026-08-22_15-54-43
folder name: /content/drive/MyDrive/VPR_megaloc_logs/2026-08-22_15-54-43/preds
Folder has something in it!
100% 1000/1000 [06:30<00:00,  2.56it/s]
R@1: 86.7, R@5: 89.6, R@10: 90.8, R@20: 91.5
Saved results to /content/drive/MyDrive/VPR_preds/logs/Rerankings/_loftr_recalls.xlsx
folder name: /content/drive/MyDrive/VPR_megaloc_logs/2026-08-22_15-54-43/preds
Folder has something in it!
100% 1000/1000 [05:41<00:00,  2.92it/s]
R@1: 85.1, R@5: 89.6, R@10: 90.9, R@20: 91.5
Saved results to /content/drive/MyDrive/VPR_preds/logs/Rerankings/_superglue_recalls.xlsx
folder name: /content/drive/MyDrive/VPR_megaloc_logs/2026-08-22_15-54-43/preds
Folder has something in it!
100% 1000/1000 [10:40<00:00,  1.56it/s]
R@1: 86.9, R@5: 90.8, R@10: 91.4, R@20: 91.5
Saved results to /content/drive/MyDrive/VPR_preds/logs/Rerankings/_superpoint-lg_recalls.xlsx

FINAL RE-RANKING SCORES (MegaLoc): 